# 🖥️ Cloud Desktop — VS Code

Run an XFCE4 desktop directly inside VS Code / GitHub Codespaces and access it via browser or VS Code's native port forwarding.

**Stack:** `Xvfb` + `XFCE4` + `x11vnc` + `noVNC` + `localhost.run` + `Brave Browser`

## 📦 Cell 1 — Install Dependencies
> Uses `sudo` for non-root VS Code environments.

In [ ]:
!sudo DEBIAN_FRONTEND=noninteractive apt-get update -qq
!sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xfce4 x11vnc xvfb dbus-x11 novnc websockify openssh-client mesa-utils pciutils 2>/dev/null
print('✅ Core packages ready!')

## 🌐 Cell 2 — Install Brave Browser

In [ ]:
%pip install torch

In [ ]:
import subprocess

try:
    res = subprocess.run('nvidia-smi', shell=True, capture_output=True, text=True)
    if res.returncode == 0:
        print('✅ NVIDIA GPU & Drivers Detected:')
        print('\n'.join(res.stdout.split('\n')[:10]))
    else:
        print('⚠️ No NVIDIA driver detected or nvidia-smi unaccessible.')
except Exception as e:
    print(f'⚠️ GPU Check Error: {e}')

try:
    import torch
    print(f'\n🔥 PyTorch CUDA Available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'🎮 Device Name: {torch.cuda.get_device_name(0)}')
except ImportError:
    print('\n💡 PyTorch not installed in current environment. Install with: !pip install torch')

## 🖥️ Cell 3 — Start Virtual Display + XFCE4

In [ ]:
import subprocess, os, time

WIDTH = 1920    # Set your width
HEIGHT = 1080   # Set your height
COLOR_DEPTH = 24

os.environ['DISPLAY'] = ':99'

xvfb_cmd = f'Xvfb :99 -screen 0 {WIDTH}x{HEIGHT}x{COLOR_DEPTH} -ac'
subprocess.Popen(xvfb_cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

subprocess.Popen('DISPLAY=:99 startxfce4', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)

print(f'✅ XFCE4 running at {WIDTH}x{HEIGHT}!')

In [1]:
import subprocess, os, time

WIDTH = 1920
HEIGHT = 1080
COLOR_DEPTH = 24

os.environ['DISPLAY'] = ':99'

# 1. Start virtual display
subprocess.Popen(f'Xvfb :99 -screen 0 {WIDTH}x{HEIGHT}x{COLOR_DEPTH} -ac', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# 2. Start system D-Bus daemon if not running
subprocess.run('sudo service dbus start', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3. Start XFCE inside an active D-Bus session
subprocess.Popen('DISPLAY=:99 dbus-launch --exit-with-session startxfce4', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)

print('✅ XFCE4 running with active D-Bus session!')

✅ XFCE4 running with active D-Bus session!


In [ ]:
import subprocess, os

# 1. Install Debian Software app & PackageKit backend
print('⏳ Installing Debian Software app...')
subprocess.run('sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gnome-software packagekit 2>/dev/null', shell=True)

# 2. Start PackageKit daemon manually (bypasses missing systemd in containers)
print('⏳ Starting PackageKit background service...')
subprocess.Popen('sudo /usr/lib/packagekit/packagekitd', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3. Launch Debian Software app on display :99 wrapped in D-Bus
print('🚀 Launching Debian Software app...')
subprocess.Popen('DISPLAY=:99 dbus-launch gnome-software', shell=True, env={**os.environ, 'DISPLAY': ':99'}, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print('✅ Debian Software app opened on your desktop!')

## 📡 Cell 4 — Start VNC Server

In [2]:
import subprocess, time, os

VNC_PASSWORD = '12345678'  # Change if you want
home_dir = os.path.expanduser('~')
pass_file = os.path.join(home_dir, '.vncpasswd')

subprocess.run(f'x11vnc -storepasswd {VNC_PASSWORD} {pass_file}', shell=True,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

subprocess.Popen(
    f'x11vnc -display :99 -rfbauth {pass_file} -rfbport 5900 -forever -loop -shared -noxdamage',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print(f'✅ VNC server running (password: {VNC_PASSWORD})')

✅ VNC server running (password: 12345678)


## 🖱️ Cell 5 — Start noVNC

In [3]:
import subprocess, time
subprocess.Popen('websockify --web=/usr/share/novnc/ 6080 localhost:5900', shell=True,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print('✅ noVNC running on port 6080!')
print('💡 VS Code Tip: You can open port 6080 directly from the VS Code Ports tab and append /vnc.html to the URL.')

✅ noVNC running on port 6080!
💡 VS Code Tip: You can open port 6080 directly from the VS Code Ports tab and append /vnc.html to the URL.


## 🔗 Cell 6 — Create Public Tunnel via localhost.run (Optional)

In [ ]:
import subprocess, time, os

log_file = os.path.expanduser('~/tunnel.log')

subprocess.Popen(
    f'ssh -o StrictHostKeyChecking=no -o ServerAliveInterval=60 -R 80:localhost:6080 nokey@localhost.run > {log_file} 2>&1',
    shell=True
)

print('⏳ Starting tunnel...')
time.sleep(15)

if os.path.exists(log_file):
    with open(log_file, 'r') as f:
        content = f.read()

    found = False
    for line in content.split('\n'):
        if 'localhost.run' in line and 'https://' in line:
            for word in line.split():
                if (word.startswith('https://') and 'localhost.run' in word
                        and not any(x in word for x in ['admin.', '/docs'])):
                    print('=' * 55)
                    print('🎉 Your remote desktop is ready!')
                    print(f'🌐 Open: {word.strip()}/vnc.html')
                    print(f'🔒 Password: {VNC_PASSWORD}')
                    print('=' * 55)
                    found = True
                    break

    if not found:
        print('⚠️ Public URL not ready — check Cell 7 or use VS Code Ports panel.')

## 🔍 Cell 7 — Show Full Tunnel Log

In [4]:
import os
log_file = os.path.expanduser('~/tunnel.log')
if os.path.exists(log_file):
    with open(log_file, 'r') as f:
        print(f.read())

Pseudo-terminal will not be allocated because stdin is not a terminal.

Welcome to localhost.run!

To set up and manage custom domains go to https://admin.localhost.run/

More details on custom domains (and how to enable subdomains of your custom
domain) at https://localhost.run/docs/custom-domains

If you get a permission denied error check the faq for how to connect with a key or
create a free tunnel without a key at [http://localhost:3000/docs/faq#generating-an-ssh-key].

To explore using localhost.run visit the documentation site:
https://localhost.run/docs/



** your connection id is 172.166.156.166:61504, please mention it if you send me a message about an issue. **

ca6522045ad544.lhr.life tunneled with tls termination, https://ca6522045ad544.lhr.life
create an account and add your key for a longer lasting domain name. see https://localhost.run/docs/forever-free/ for more information.
Open your tunnel address on your mobile with this QR:

                                       

## 🚀 Cell 8 — Launch Brave Browser on the Desktop

In [ ]:
import subprocess, os
subprocess.Popen('DISPLAY=:99 brave-browser --no-sandbox', shell=True,
    env={**os.environ, 'DISPLAY': ':99'},
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('✅ Brave launched on the desktop!')

In [ ]:
import subprocess

# 1. Set System Timezone to Europe/London
print("⏳ Setting timezone to Europe/London...")
subprocess.run("sudo ln -sf /usr/share/zoneinfo/Europe/London /etc/localtime", shell=True)
subprocess.run("sudo dpkg-reconfigure -f noninteractive tzdata", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 2. Set Default System Keyboard to UK (gb)
keyboard_config = """XKBMODEL="pc105"
XKBLAYOUT="gb"
XKBVARIANT=""
XKBOPTIONS=""
BACKSPACE="guess"
"""
with open("/tmp/keyboard", "w") as f:
    f.write(keyboard_config)
subprocess.run("sudo mv /tmp/keyboard /etc/default/keyboard", shell=True)

# 3. Apply UK layout immediately to the running display (:99)
subprocess.run("DISPLAY=:99 setxkbmap gb", shell=True, stderr=subprocess.DEVNULL)

print("✅ System defaults updated! Timezone is now Europe/London and keyboard layout is set to UK.")

In [ ]:
import subprocess

# 1. Create a legacy PKLA rule (for Debian/Ubuntu Polkit 0.105)
pkla_content = """[Allow All Actions]
Identity=unix-user:*
Action=*
ResultAny=yes
ResultInactive=yes
ResultActive=yes
"""

subprocess.run("sudo mkdir -p /etc/polkit-1/localauthority/50-local.d/", shell=True)
with open("/tmp/45-allow-all.pkla", "w") as f:
    f.write(pkla_content)
subprocess.run("sudo mv /tmp/45-allow-all.pkla /etc/polkit-1/localauthority/50-local.d/45-allow-all.pkla", shell=True)

# 2. Create a JS rule (for newer Polkit versions)
js_rule_content = """polkit.addRule(function(action, subject) {
    return polkit.Result.YES;
});
"""

subprocess.run("sudo mkdir -p /etc/polkit-1/rules.d/", shell=True)
with open("/tmp/45-allow-all.rules", "w") as f:
    f.write(js_rule_content)
subprocess.run("sudo mv /tmp/45-allow-all.rules /etc/polkit-1/rules.d/45-allow-all.rules", shell=True)

# 3. Restart Polkit service to apply changes
subprocess.run("sudo service polkit restart", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("✅ Auto-authentication enabled! Password prompts will no longer appear.")

In [ ]:
import subprocess

# 1. Enable 32-bit architecture required by Steam
subprocess.run("sudo dpkg --add-architecture i386", shell=True)

# 2. Update package list with new architecture
subprocess.run("sudo DEBIAN_FRONTEND=noninteractive apt-get update -qq", shell=True)

# 3. Install Steam and 32-bit graphics/system dependencies
steam_packages = (
    "steam "
    "libc6:i386 "
    "libgl1-mesa-dri:i386 "
    "libgl1-mesa-glx:i386 "
    "libglx-mesa0:i386 "
    "mesa-vulkan-drivers "
    "mesa-vulkan-drivers:i386 "
    "libgpg-error0:i386 "
    "libdbus-1-3:i386"
)

print("⏳ Installing Steam and 32-bit architecture libraries...")
subprocess.run(f"sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq {steam_packages} 2>/dev/null", shell=True)
print("✅ Steam dependencies successfully installed!")

In [ ]:
import subprocess, os

# 1. Install PulseAudio, ALSA utilities, and XFCE audio controls
print("⏳ Installing PulseAudio and volume control tools...")
subprocess.run(
    "sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
    "pulseaudio pavucontrol xfce4-pulseaudio-plugin alsa-utils 2>/dev/null",
    shell=True
)

# 2. Start PulseAudio daemon in background with a virtual dummy output
print("⏳ Initializing PulseAudio virtual audio sink...")
subprocess.run("pulseaudio --kill", shell=True, stderr=subprocess.DEVNULL)
subprocess.Popen(
    "pulseaudio --start --realtime=no --exit-idle-time=-1",
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# 3. Create a dummy virtual audio output device so apps can play sound
subprocess.run(
    "pacmd load-module module-null-sink sink_name=Virtual_Sink sink_properties=device.description='Virtual_Audio_Output'",
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
subprocess.run(
    "pacmd set-default-sink Virtual_Sink",
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# 4. Launch Pavucontrol (Volume Control GUI) onto the desktop
print("🚀 Opening Volume Control GUI on your desktop...")
subprocess.Popen(
    "DISPLAY=:99 pavucontrol",
    shell=True, env={**os.environ, "DISPLAY": ":99"},
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print("✅ Audio control system initialized!")

In [ ]:
import subprocess, time, os

# 1. Install ffmpeg for live audio encoding
print("⏳ Installing ffmpeg...")
subprocess.run("sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ffmpeg 2>/dev/null", shell=True)

# 2. Create the python audio streaming server script
server_script = """import http.server
import socketserver
import subprocess

PORT = 8000

class AudioStreamHandler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200)
        self.send_header('Content-Type', 'audio/mpeg')
        self.send_header('Cache-Control', 'no-cache, no-store, must-revalidate')
        self.send_header('Pragma', 'no-cache')
        self.send_header('Expires', '0')
        self.end_headers()
        
        # Capture system PulseAudio output and convert to live MP3 stream
        cmd = ['ffmpeg', '-f', 'pulse', '-i', 'default', '-acodec', 'libmp3lame', '-ab', '128k', '-f', 'mp3', '-']
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        
        try:
            while True:
                chunk = proc.stdout.read(4096)
                if not chunk:
                    break
                self.wfile.write(chunk)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            proc.terminate()

socketserver.TCPServer.allow_reuse_address = True
with socketserver.TCPServer(("", PORT), AudioStreamHandler) as httpd:
    httpd.serve_forever()
"""

with open("/tmp/audio_server.py", "w") as f:
    f.write(server_script)

# 3. Kill any existing server on port 8000 and start new background audio server
subprocess.run("fuser -k 8000/tcp", shell=True, stderr=subprocess.DEVNULL)
subprocess.Popen("python3 /tmp/audio_server.py", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

print("✅ Live Audio Streamer started on port 8000!")

In [ ]:
import subprocess
subprocess.run('pkill x11vnc; pkill Xvfb; pkill xfce4-session; pkill websockify; pkill ssh', shell=True)
print('🛑 All processes stopped!')

In [ ]:
import subprocess

print("📊 Current Disk Usage:")
subprocess.run("df -h /", shell=True)

print("\n🧹 Cleaning APT package cache & unused dependencies...")
subprocess.run("sudo apt-get clean", shell=True)
subprocess.run("sudo apt-get autoremove -y -qq", shell=True)

print("🧹 Cleaning Python pip cache...")
subprocess.run("pip cache purge", shell=True)

print("🧹 Clearing temporary files & browser/thumbnail caches...")
subprocess.run("rm -rf ~/.cache/thumbnails/* /tmp/* 2>/dev/null", shell=True)
subprocess.run("rm -rf ~/.cache/BraveSoftware/Brave-Browser/Default/Cache/* 2>/dev/null", shell=True)

# Optional: Clean HuggingFace cache if you downloaded large models
subprocess.run("rm -rf ~/.cache/huggingface 2>/dev/null", shell=True)

print("\n✅ Cleanup complete! Updated Disk Usage:")
subprocess.run("df -h /", shell=True)

In [ ]:
import subprocess, os

# 1. Install rclone
print("⏳ Installing rclone...")
subprocess.run("sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq rclone 2>/dev/null", shell=True)

# 2. Create local mount directory
mount_path = os.path.expanduser("~/GoogleDrive")
os.makedirs(mount_path, exist_ok=True)

print(f"✅ rclone installed! Mount point created at: {mount_path}")

In [ ]:
import subprocess, os

mount_path = os.path.expanduser("~/GoogleDrive")
os.makedirs(mount_path, exist_ok=True)

# Unmount if previously active
subprocess.run(f"fusermount -u {mount_path}", shell=True, stderr=subprocess.DEVNULL)

# Mount drive in background
mount_cmd = (
    f"rclone mount gdrive: {mount_path} "
    "--vfs-cache-mode full "
    "--allow-other &"
)

subprocess.Popen(mount_cmd, shell=True)
print(f"🎉 Google Drive successfully mounted at {mount_path}!")